<a href="https://colab.research.google.com/github/hyperpipe-kr/colab-examples/blob/main/07_RAG_web_QA_%EC%98%88%EC%A0%9C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 환경변수 파일(env) 설정하기
/content/drive/MyDrive/lg-dx/에 아래 내용으로 env 파일을 작성한 후 업로드합니다.  

OPENAI_API_KEY=sk-xxxx  
LANGCHAIN_TRACING_V2=true  
LANGCHAIN_ENDPOINT=https://api.smith.langchain.com  
LANGCHAIN_API_KEY=lsv2_xxxx  
LANGCHAIN_PROJECT=lg-dx


Google Drive 마운트

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


python-dotenv 라이브러리 설치

In [2]:
!pip install python-dotenv

환경변수 파일 로드 및 확인

In [ ]:
from dotenv import load_dotenv
load_dotenv('/content/drive/MyDrive/lg-dx/env', override=True)

import os
openai_api_key = os.environ.get('OPENAI_API_KEY')
print("openai_api_key : " + openai_api_key)
langchain_api_key = os.environ.get('LANGCHAIN_API_KEY')
print("langchain_api_key : " + langchain_api_key)

## LangChain 설치 및 업데이트

In [5]:
# LangChain 설치 및 업데이트
!pip install -qU langchain-teddynote langchain_community pymupdf faiss-cpu

설치된 LangChain 버전을 확인합니다.

In [6]:
from importlib.metadata import version

print("[LangChain 관련 패키지 버전]")
for package_name in [
    "langchain",
    "langchain-core",
    "langchain-experimental",
    "langchain-community",
    "langchain-openai",
    "langchain-teddynote",
    "langchain-huggingface",
    "langchain-google-genai",
    "langchain-anthropic",
    "langchain-cohere",
    "langchain-chroma",
    "langchain-elasticsearch",
    "langchain-upstage",
    "langchain-cohere",
    "langchain-milvus",
    "langchain-text-splitters",
]:
    try:
        package_version = version(package_name)
        print(f"{package_name}: {package_version}")
    except ImportError:
        print(f"{package_name}: 설치되지 않음")

[LangChain 관련 패키지 버전]
langchain: 1.3.13
langchain-core: 1.5.1
langchain-experimental: 설치되지 않음
langchain-community: 0.4.2
langchain-openai: 1.4.1
langchain-teddynote: 0.5.4
langchain-huggingface: 설치되지 않음
langchain-google-genai: 설치되지 않음
langchain-anthropic: 설치되지 않음
langchain-cohere: 설치되지 않음
langchain-chroma: 설치되지 않음
langchain-elasticsearch: 설치되지 않음
langchain-upstage: 설치되지 않음
langchain-cohere: 설치되지 않음
langchain-milvus: 설치되지 않음
langchain-text-splitters: 1.1.2


LangSmith 추적 설정  
.env 파일에 LangSmith API 키가 설정 되어 있어야 합니다.(LANGCHAIN_API_KEY)

In [7]:
from langchain_teddynote import logging

# 프로젝트 이름을 입력합니다.
logging.langsmith("lg-dx-rag-web")

LangSmith 추적을 시작합니다.
[프로젝트명]
lg-dx-rag-web


## 네이버 뉴스 기반 QA(Question-Answering) 챗봇
이번 튜토리얼에는 네이버 뉴스기사의 내용에 대해 질문할 수 있는 뉴스기사 QA 앱 을 구축할 것입니다.  

이 가이드에서는 OpenAI 챗 모델과 임베딩, 그리고 Chroma 벡터 스토어를 사용할 것입니다.  

먼저 다음의 과정을 통해 간단한 인덱싱 파이프라인과 RAG 체인을 약 20줄의 코드로 구현할 수 있습니다.
- bs4는 웹 페이지를 파싱하기 위한 라이브러리입니다.
- langchain은 AI와 관련된 다양한 기능을 제공하는 라이브러리로, 여기서는 특히 텍스트 분할(RecursiveCharacterTextSplitter), 문서 로딩(WebBaseLoader), 벡터 저장(Chroma, FAISS), 출력 파싱(StrOutputParser), 실행 가능한 패스스루(RunnablePassthrough) 등을 다룹니다.  
- langchain_openai 모듈을 통해 OpenAI의 챗봇(ChatOpenAI)과 임베딩(OpenAIEmbeddings) 기능을 사용할 수 있습니다.


In [8]:
import bs4
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

/tmp/ipykernel_4546/746077902.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader


## RAG 기본 파이프라인(1~8단계)

In [9]:
import bs4
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

In [10]:
# 단계 1: 문서 로드(Load Documents)
# 뉴스기사 내용을 로드하고, 청크로 나누고, 인덱싱합니다.
loader = WebBaseLoader(
    web_paths=("https://n.news.naver.com/article/437/0000378416",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            "div",
            attrs={"class": ["newsct_article _article_body", "media_end_head_title"]},
        )
    ),
)

docs = loader.load()
print(f"문서의 수: {len(docs)}")
docs

문서의 수: 1


[Document(metadata={'source': 'https://n.news.naver.com/article/437/0000378416'}, page_content="\n출산 직원에게 '1억원' 쏜다…회사의 파격적 저출생 정책\n\n\n[앵커]올해 아이 낳을 계획이 있는 가족이라면 솔깃할 소식입니다. 정부가 저출생 대책으로 매달 주는 부모 급여, 0세 아이는 100만원으로 올렸습니다. 여기에 첫만남이용권, 아동수당까지 더하면 아이 돌까지 1년 동안 1520만원을 받습니다. 지자체도 경쟁하듯 지원에 나섰습니다. 인천시는 새로 태어난 아기, 18살될 때까지 1억원을 주겠다. 광주시도 17살될 때까지 7400만원 주겠다고 했습니다. 선거 때면 나타나서 아이 낳으면 현금 주겠다고 밝힌 사람이 있었죠. 과거에는 표만 노린 '황당 공약'이라는 비판이 따라다녔습니다. 그런데 지금은 출산율이 이보다 더 나쁠 수 없다보니, 이런 현금성 지원을 진지하게 정책화 하는 상황까지 온 겁니다. 게다가 기업들도 뛰어들고 있습니다. 이번에는 출산한 직원에게 단번에 1억원을 주겠다는 회사까지 나타났습니다.이상화 기자가 취재했습니다.[기자]한 그룹사가 오늘 파격적인 저출생 정책을 내놨습니다.2021년 이후 태어난 직원 자녀에 1억원씩, 총 70억원을 지원하고 앞으로도 이 정책을 이어가기로 했습니다.해당 기간에 연년생과 쌍둥이 자녀가 있으면 총 2억원을 받게 됩니다.[오현석/부영그룹 직원 : 아이 키우는 데 금전적으로 많이 힘든 세상이잖아요. 교육이나 생활하는 데 큰 도움이 될 거라 생각합니다.]만약 셋째까지 낳는 경우엔 국민주택을 제공하겠다는 뜻도 밝혔습니다.[이중근/부영그룹 회장 : 3년 이내에 세 아이를 갖는 분이 나올 것이고 따라서 주택을 제공할 수 있는 계기가 될 것으로 생각하고.][조용현/부영그룹 직원 : 와이프가 셋째도 갖고 싶어 했는데 경제적 부담 때문에 부정적이었거든요. (이제) 긍정적으로 생각할 수 있을 것 같습니다.]오늘 행사에서는, 회사가 제공하는 출산장려금은 받는 

In [11]:
# 단계 2: 문서 분할(Split Documents)
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)

splits = text_splitter.split_documents(docs)
len(splits)

3

In [12]:
# 단계 3: 임베딩(Embedding) 생성
embeddings = OpenAIEmbeddings()

# 단계 4: DB 생성(Create DB) 및 저장
# 벡터스토어를 생성합니다.
vectorstore = FAISS.from_documents(documents=splits, embedding=embeddings)

In [13]:
# 단계 5: 검색기(Retriever) 생성
# 문서에 포함되어 있는 정보를 검색하고 생성합니다.
# 검색 결과 4개(default)
retriever = vectorstore.as_retriever()

# 검색 결과 8개
# retriever = vectorstore.as_retriever(search_kwargs={"k": 8})


In [14]:
# 단계 6: 프롬프트 생성(Create Prompt)
# 프롬프트를 생성합니다.
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template(
    """당신은 질문-답변(Question-Answering)을 수행하는 친절한 AI 어시스턴트입니다. 당신의 임무는 주어진 문맥(context) 에서 주어진 질문(question) 에 답하는 것입니다.
검색된 다음 문맥(context) 을 사용하여 질문(question) 에 답하세요. 만약, 주어진 문맥(context) 에서 답을 찾을 수 없다면, 답을 모른다면 `주어진 정보에서 질문에 대한 정보를 찾을 수 없습니다` 라고 답하세요.
한글로 답변해 주세요. 단, 기술적인 용어나 이름은 번역하지 않고 그대로 사용해 주세요.

#Question:
{question}

#Context:
{context}

#Answer:"""
)


In [18]:
# 단계 7: 언어모델(LLM) 생성
# 모델(LLM) 을 생성합니다.
llm = ChatOpenAI(model_name="gpt-5.4-mini", temperature=0)

In [19]:
# 단계 8: 체인(Chain) 생성
chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [20]:
# 체인 실행(Run Chain)
# 문서에 대한 질의를 입력하고, 답변을 출력합니다.
answer = chain.invoke("부영그룹의 출산 장려 정책에 대해 설명해주세요.")
print(answer)

부영그룹은 **2021년 이후 태어난 직원 자녀에게 1명당 1억원씩 출산장려금을 지급**하는 정책을 내놨습니다. 이 정책에 따라 **총 70억원**을 지원하고, 앞으로도 계속 이어가기로 했습니다.  
또한 **연년생이나 쌍둥이 자녀가 있으면 총 2억원**을 받게 됩니다.

추가로, **셋째를 낳는 경우에는 국민주택을 제공하겠다는 방침**도 밝혔습니다.  
즉, 부영그룹의 출산 장려 정책은 **현금 지원과 주택 제공을 통해 직원들의 출산 부담을 줄이려는 것**입니다.


In [21]:
answer = chain.invoke("부영그룹은 출산 직원에게 얼마의 지원을 제공하나요?")
print(answer)

부영그룹은 출산한 직원의 자녀 1명당 **1억원**을 지원합니다.


In [22]:
answer = chain.invoke("정부의 저출생 대책을 bullet points 형식으로 작성해 주세요.")
print(answer)

- 정부가 저출생 대책으로 매달 주는 **부모 급여**를 올렸습니다.
- **0세 아이의 부모 급여를 100만원**으로 인상했습니다.
- **첫만남이용권**을 지원합니다.
- **아동수당**을 지원합니다.
- 이 지원들을 합치면 **아이 돌까지 1년 동안 1520만원**을 받을 수 있습니다.


In [23]:
answer = chain.invoke("부영그룹의 임직원 숫자는 몇명인가요?")
print(answer)

주어진 정보에서 질문에 대한 정보를 찾을 수 없습니다
